In [ ]:
import logging
import re # Keep re for potential future use in helpers
from collections import Counter
from itertools import chain
from typing import Any, Dict, List, Optional, Sequence

# Basic logger setup
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class Tokenizer:
    """
    Vocabulary and tokenizer based on word frequency.

    Builds a vocabulary from a corpus file, filtering infrequent words
    and reserving indices for standard and custom special tokens.
    """
    word2idx: Dict[str, int]
    idx2word: Dict[int, str]
    special_tokens: Dict[str, int]
    vocab_size: int
    num_special_tokens: int
    bos_id: int
    eos_id: int
    pad_id: int
    unk_token_id: int
    nhi_token_id: int # Custom token
    nhpr_token_id: int # Custom token

    def __init__(self, corpus_file: str, vocab_size: int = 3000, min_freq: int = 3):
        """
        Initializes the Tokenizer.

        Args:
            corpus_file (str): Path to the text file containing pre-processed
                               captions/reports, one per line, whitespace-separated words.
            vocab_size (int): The target total size of the vocabulary, including
                              all special tokens.
            min_freq (int): The minimum frequency for a word to be included
                            in the vocabulary (before applying vocab_size limit).
        """
        logger.info(f"Initializing Tokenizer from corpus: {corpus_file}")
        logger.info(f"Target vocabulary size: {vocab_size}, Minimum word frequency: {min_freq}")

        # --- Define Special Tokens (using more standard names where applicable) ---
        self.special_tokens = {
            "<pad>": 0,      # Padding token
            "<bos>": 1,      # Begin-Of-Sequence token (equivalent to <start>)
            "<eos>": 2,      # End-Of-Sequence token (equivalent to <end>)
            "<unk>": 3,      # Unknown word token
            "[NHI]": 4,      # Custom: No Header / Indication placeholder
            "[NHPR]": 5,     # Custom: No Prior Report placeholder
        }
        self.num_special_tokens = len(self.special_tokens)

        # Assign common attribute names for key token IDs
        self.pad_id = self.special_tokens["<pad>"]
        self.bos_id = self.special_tokens["<bos>"]
        self.eos_id = self.special_tokens["<eos>"]
        self.unk_token_id = self.special_tokens["<unk>"]
        # Assign IDs for custom tokens if needed elsewhere
        self.nhi_token_id = self.special_tokens["[NHI]"]
        self.nhpr_token_id = self.special_tokens["[NHPR]"]

        # Ensure vocab_size is large enough for special tokens
        if vocab_size < self.num_special_tokens:
            raise ValueError(f"vocab_size ({vocab_size}) must be at least {self.num_special_tokens} "
                             f"to accommodate all special tokens.")

        # --- Build Vocabulary from Corpus ---
        try:
            with open(corpus_file, 'r', encoding='utf-8') as f:
                # Assume corpus file contains already cleaned/pre-processed text
                lines = [line.strip() for line in f if line.strip()]
        except FileNotFoundError:
            logger.error(f"Corpus file not found: {corpus_file}")
            raise
        except Exception as e:
            logger.error(f"Error reading corpus file {corpus_file}: {e}")
            raise

        if not lines:
            logger.warning(f"Corpus file {corpus_file} is empty or contains only whitespace.")
            # Initialize with only special tokens
            self.word2idx = self.special_tokens.copy()
            self.idx2word = {idx: word for word, idx in self.word2idx.items()}
            self.vocab_size = self.num_special_tokens
            logger.info(f"Initialized tokenizer with {self.vocab_size} special tokens only.")
            return

        # Calculate word frequency from space-separated words in lines
        logger.info("Calculating word frequencies...")
        # Use simple space splitting, assuming pre-processing handled complex cases
        word_freq = Counter(chain.from_iterable(line.split(' ') for line in lines))
        logger.info(f"Found {len(word_freq)} unique words initially.")

        # Filter words by minimum frequency
        filtered_word_freq = {word: freq for word, freq in word_freq.items() if freq >= min_freq}
        logger.info(f"Kept {len(filtered_word_freq)} words after filtering (min_freq={min_freq}).")

        # Calculate how many words to keep based on target vocab size
        num_words_to_keep = vocab_size - self.num_special_tokens

        # Get the most common words from the filtered list
        # Use Counter for efficient most_common
        most_common_words = Counter(filtered_word_freq).most_common(num_words_to_keep)
        logger.info(f"Selected {len(most_common_words)} most common words to fit target vocab size.")

        # --- Create Mappings ---
        # Start indexing corpus words *after* the special tokens
        self.word2idx = {word: i + self.num_special_tokens for i, (word, _) in enumerate(most_common_words)}
        # Add the special tokens with their predefined indices
        self.word2idx.update(self.special_tokens)

        # Create the reverse mapping
        self.idx2word = {idx: word for word, idx in self.word2idx.items()}

        # Store the actual final vocabulary size
        self.vocab_size = len(self.word2idx)
        logger.info(f"Final vocabulary size: {self.vocab_size}")
        if self.vocab_size < vocab_size:
             logger.warning(f"Actual vocabulary size ({self.vocab_size}) is less than target ({vocab_size}) "
                            f"due to filtering and available words.")

        # Store vocab size consistent with TikToken naming convention
        self.n_words = self.vocab_size


    def encode(self, text: str, bos: bool = True, eos: bool = True) -> List[int]:
        """
        Converts a text string into a list of token indices.
        Assumes text is pre-processed and words are space-separated.

        Args:
            text (str): The input text string.
            bos (bool): Whether to prepend the beginning-of-sequence token (<bos>).
                        Defaults to True.
            eos (bool): Whether to append the end-of-sequence token (<eos>).
                        Defaults to True.

        Returns:
            List[int]: A list of token indices corresponding to the text.
        """
        if not isinstance(text, str):
             logger.warning(f"Input to encode is not a string: {type(text)}. Returning empty list.")
             return []

        # Simple whitespace splitting
        tokens = text.split(' ')
        # Filter out potential empty strings resulting from multiple spaces
        tokens = [token for token in tokens if token]

        indices = []
        if bos:
            indices.append(self.bos_id)

        # Map words to indices, using <unk> for out-of-vocabulary words
        indices.extend(self.word2idx.get(word, self.unk_token_id) for word in tokens)

        if eos:
            indices.append(self.eos_id)

        return indices

    def decode(self, indices: Sequence[int], skip_special_tokens: bool = True) -> str:
        """
        Converts a sequence of token indices back into a text string.

        Args:
            indices (Sequence[int]): The sequence of token indices.
            skip_special_tokens (bool): Whether to skip all special tokens
                                        (indices < num_special_tokens) during decoding.
                                        Defaults to True.

        Returns:
            str: The decoded text string.
        """
        if not isinstance(indices, (list, tuple)): # Check Sequence types
            logger.warning(f"Input to decode is not a sequence (list/tuple): {type(indices)}. Returning empty string.")
            return ""

        words = []
        for idx in indices:
            # Use .get() for robustness against invalid indices
            word = self.idx2word.get(idx, "<unk>")
            if skip_special_tokens and idx < self.num_special_tokens:
                continue # Skip if it's a special token and we're skipping
            words.append(word)

        return " ".join(words) # Join words with spaces

    def __len__(self) -> int:
        """Returns the size of the vocabulary (consistent with n_words)."""
        return self.vocab_size

# --- Example Usage ---
if __name__ == "__main__":
    # Create a dummy corpus file (assume it's already pre-processed)
    dummy_corpus = "dummy_corpus_cleaned.txt"
    with open(dummy_corpus, "w", encoding='utf-8') as f:
        f.write("this is the first report with normal findings\n")
        f.write("second report shows mild disease\n")
        f.write("the patient has normal lungs\n")
        f.write("mild findings are noted again\n")
        f.write("report indicates disease\n")
        f.write("normal normal normal\n") # Test frequency filter
        f.write("unique word here\n") # Test frequency filter
        f.write("[NHI] placeholder text\n") # Treat as regular words if not pre-filtered

    # Initialize tokenizer
    tokenizer = Tokenizer(dummy_corpus, vocab_size=15, min_freq=2) # Target 15, min freq 2

    print("\nVocabulary (word to index):")
    print(tokenizer.word2idx)
    print("\nVocabulary (index to word):")
    print(tokenizer.idx2word)
    print(f"\nVocabulary Size (n_words): {tokenizer.n_words}")
    print(f"BOS ID: {tokenizer.bos_id}, EOS ID: {tokenizer.eos_id}, PAD ID: {tokenizer.pad_id}, UNK ID: {tokenizer.unk_token_id}")
    print(f"Custom [NHI] ID: {tokenizer.nhi_token_id}, [NHPR] ID: {tokenizer.nhpr_token_id}")


    # Test encoding
    text_to_encode = "this report shows new findings and [NHI]"
    # Encode with default bos=True, eos=True
    encoded = tokenizer.encode(text_to_encode)
    print(f"\nEncoding '{text_to_encode}' (bos=T, eos=T):")
    print(encoded) # Should include <unk> for 'new' and map '[NHI]' if in vocab

    # Encode without bos/eos
    encoded_no_special = tokenizer.encode(text_to_encode, bos=False, eos=False)
    print(f"\nEncoding '{text_to_encode}' (bos=F, eos=F):")
    print(encoded_no_special)

    # Test decoding
    decoded_skip = tokenizer.decode(encoded) # Default skip_special_tokens=True
    print(f"\nDecoding {encoded} (skip special):")
    print(decoded_skip) # Should omit <bos>, <eos>, <pad>, <unk> if present

    decoded_no_skip = tokenizer.decode(encoded, skip_special_tokens=False) # Keep special tokens
    print(f"\nDecoding {encoded} (keep special):")
    print(decoded_no_skip) # Should include the special token strings

    # Test decoding sequence without bos/eos
    decoded_no_special_skip = tokenizer.decode(encoded_no_special)
    print(f"\nDecoding {encoded_no_special} (skip special):")
    print(decoded_no_special_skip)

    # Clean up dummy file
    import os
    os.remove(dummy_corpus)

In [1]:
import torch
import sys
from config import Config
from data import Tokenizer
from data.build_corpus import build_corpus
from data.utils import train_val_test_split, load_torch_dataloaders, load_torch_dataset
from models import MedicalReportGenerator
from training.train import train
from utils.compute_params import compute_parameters
from training.metrics import compute_metrics
from training.evaluate import evaluate_model
from utils.write_to_csv import write_to_csv
from utils.compute_flops import compute_flops

dataset_name = 'mimic-cxr'
print("Loading the configuration for dataset: ", dataset_name)
config = Config(dataset_name)

# Build the corpus file
report_df, output_file = build_corpus(config)
tokenizer = Tokenizer(corpus_file=output_file, vocab_size=config.vocab_size, min_freq=config.min_word_freq)

# Get the train_test_data
train_data, val_data, test_data = train_val_test_split(config, report_df)

# Load the pytorch dataset
# train_dataset, val_dataset, test_dataset = load_torch_dataset(config, tokenizer, train_data, val_data, test_data)
train_dataset, val_dataset, test_dataset = load_torch_dataset(config, tokenizer, train_data[:7], val_data[:1], test_data[:2]) # debugging with few samples

# Initialize the dataloaders
train_loader, val_loader, test_loader = load_torch_dataloaders(config, train_dataset, val_dataset, test_dataset)

# Train the model
model = MedicalReportGenerator(config, tokenizer).to(config.device)

train(model, train_loader, val_loader, test_loader, config, tokenizer)

/home/users/nshaik3/miniconda3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to
[nltk_data]     /home/users/nshaik3/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
2025-09-08 16:49:41,656 - data.build_corpus - INFO - Starting corpus building process...
INFO:data.build_corpus:Starting corpus building process...
2025-09-08 16:49:41,658 - data.build_corpus - INFO - Loading data using cfg.data_prep...
INFO:data.build_corpus:Loading data using cfg.data_prep...


Loading the configuration for dataset:  mimic-cxr


2025-09-08 16:49:44,189 - data.build_corpus - INFO - Data loaded successfully. Shape: (95169, 23)
INFO:data.build_corpus:Data loaded successfully. Shape: (95169, 23)
2025-09-08 16:49:44,192 - data.build_corpus - INFO - Processing column: 'prior_indication'...
INFO:data.build_corpus:Processing column: 'prior_indication'...
2025-09-08 16:49:45,239 - data.build_corpus - INFO - Finished processing column: 'prior_indication'.
INFO:data.build_corpus:Finished processing column: 'prior_indication'.
2025-09-08 16:49:45,241 - data.build_corpus - INFO - Processing column: 'prior_findings'...
INFO:data.build_corpus:Processing column: 'prior_findings'...
2025-09-08 16:49:50,896 - data.build_corpus - INFO - Finished processing column: 'prior_findings'.
INFO:data.build_corpus:Finished processing column: 'prior_findings'.
2025-09-08 16:49:50,900 - data.build_corpus - INFO - Processing column: 'prior_impression'...
INFO:data.build_corpus:Processing column: 'prior_impression'...
2025-09-08 16:49:52,569 

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.69 GiB. GPU 

In [ ]:
# Load the best model from checkpoint
config.load_checkpoint(model)

# Compute parameters
trainable, non_trainable, total = compute_parameters(model)
print(f"Trainable Parameters: {trainable}")
print(f"Non-Trainable Parameters: {non_trainable}")
print(f"Total Parameters: {total}")

references, hypotheses, actual_predicted_samples = evaluate_model(model, config, test_loader)

write_to_csv(actual_predicted_samples, config.results_file_save_path)

compute_metrics(references, hypotheses, actual_predicted_samples)

flops = compute_flops(model, config)
print("FLOPs: ", flops / 1e9, "GFLOPs")